# 第 7 週 Part B：診所掛號系統（個人作品）

**規則：** 個人獨立完成；可以使用任何 AI 工具；**禁止與其他同學討論**。

**流程（照順序做）：**
1. 在下一格寫好**規格書**（可以請 AI 幫忙寫初稿，但你要讀過並修改成你要的樣子）。
2. 把規格書交給 AI，請它依照規格產生程式，貼到下面對應的格子。
3. 執行**測試清單**，看懂每一句的 AI 原始回覆與系統解讀。
4. 照最後一格的**交件前檢查清單**交件。

**系統需求（簡化版）：** 科別只有內科、外科、牙科，每科每天有固定名額。功能有掛號、取消、查詢剩餘名額。
🔴 **安全規定：系統不可以提供任何醫療建議或根據症狀推薦科別。** 使用者沒說出科別時，請他直接說要掛哪一科，或洽詢櫃台。

# 📋 規格書（請直接在這一格修改）

學號：＿＿＿411303148＿＿＿　姓名：＿＿＿吳菀秦＿＿＿

## 1. 系統目的（一句話）
例：讓民眾用一句話完成診所的掛號、取消與名額查詢。

## 2. 使用者會怎麼說（至少 5 句）
1.請問現在可以掛號嗎?
2.有推薦哪一位醫生嗎?
3.我要預約早上11點洗牙
4.你好，我要取消明天的預約
5.查詢內科還有多少名額

## 3. 資料表（最多 2 張）
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| dept | subject | integer | 科別名稱 |
| dept | quota | integer | 最大看診數量 |

| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| record | id | integer | 看診編號 |
| record | name | text | 患者名稱 |
| record | dept | text | 預約科別 |

初始資料：dept:內科、外科、牙科

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,對象`（例：`掛號,內科`）
- 動作只能是：掛號、取消掛號、查詢名額
- 對象只能是：內科、外科、牙科
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：提供任何醫療建議、根據患者症狀猜測或推薦科別、解釋病情

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 掛號 | 科別 booked_count < quota | 掛號成功，顯示分配到的看診編號 | registrations 新增一筆記錄，對應科別 booked_count +1 |
| 掛號 | 科別 booked_count >= quota | 掛號失敗，該科別今日名額已滿 | 資料庫無變化 |
| 取消掛號 | 存在該科別且狀態為有效的掛號記錄 | 取消成功，該預約已作廢 | 對應記錄狀態改為「已取消」，對應科別 booked_count -1 |
| 查詢名額 | 指定科別存在 | 顯示該科別的剩餘名額（quota - booked_count） | 資料庫無變化 |

## 6. 例外情況（至少 3 個）
1.使用者輸入的科別不在內科、外科、牙科範圍內（例如：眼科、皮膚科）
2.使用者僅描述身體症狀（例如：「我肚子痛、頭暈」），未明確說出科別
3.該科別當日名額已滿，使用者仍送出掛號請求

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 我要掛內科 | AI 輸出 掛號,內科，系統回應內科掛號成功並給予編號 |
| 2 | 正常 | 查詢外科剩餘名額 | AI 輸出 查詢名額,外科，系統顯示外科目前剩餘名額數 |
| 3 | 正常 | 取消我的牙科預約 | AI 輸出 取消掛號,牙科，系統回應牙科預約已成功取消 |
| 4 | 錯誤 | 我要掛眼科 | AI 輸出 掛號,未知，系統回應查無此科別，請輸入內科、外科或牙科 |
| 5 | 錯誤 | 我最近一直咳嗽、胸悶，該看什麼科？ | AI 輸出 未知,未知，系統回應無法辨識動作與科別，且系統不提供醫療建議 |
| 6 | 錯誤 | 幫我取消內科預約（但無先前預約記錄） | AI 輸出 取消掛號,內科，系統回應找不到您的有效預約記錄，無法取消 |
| 7 | 邊界 | 內科名額已滿（剩餘 0 名）時輸入「我要掛內科」 | AI 輸出 掛號,內科，系統回應內科今日名額已滿，掛號失敗 |
| 8 | 邊界 | 內科僅剩最後 1 個名額時輸入「我要掛內科」 | AI 輸出 掛號,內科，系統回應內科掛號成功，且該科別名額歸零 |

## 🤖 可以使用的 prompt

**① 請 AI 寫規格書初稿**
```
我要做一個「診所掛號系統」的課堂小作品（Google Colab + SQLite + Groq + Gradio）。
請依照下面的規格範本，幫我寫出一份規格書初稿。要求：
1. 科別只有：內科、外科、牙科，每科每天有固定名額。
2. 功能：掛號、取消掛號、查詢某科剩餘名額。
3. 資料表不超過 2 張。
4. AI 只負責判斷「動作」與「科別」。如果使用者只描述症狀、沒有說出科別，AI 一律回答未知；系統絕對不可以提供任何醫療建議或推薦科別。
5. 測試清單至少 8 句，要包含正常、錯誤、邊界三種情況，並寫出預期結果。
規格範本如下：
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫一份 Google Colab 程式，並遵守以下要求：
1. 第 1 格（安裝套件與讀取金鑰）我已經有了，請從第 2 格開始寫，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) 測試清單 (6) 功能函式 (7) 主流程與完整流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經定義好 client（Groq）與 MODEL，也已經 import 了 sqlite3、pandas as pd、display、gradio as gr，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 temperature=0，並用 try/except 包起來，失敗時回傳「未知」，不可讓程式當掉。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」。
7. fetchone() 之後一定要先檢查是否為 None。
8. 資料庫連線用 try/finally 確保關閉。
9. 測試清單的格式為 (使用者說的話, 預期動作, 預期科別)，跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 只用 sqlite3、groq、gradio、pandas，不要用其他套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (2) 建立資料庫與初始資料
import sqlite3

def init_db():
    """
    建立 clinic.db 資料庫，並包含 dept（科別與名額）與 record（掛號記錄）兩張資料表。
    每次執行都會先 DROP TABLE 確保環境乾淨，並寫入初始資料。
    """
    conn = sqlite3.connect("clinic.db")
    cursor = conn.cursor()
    try:
        # 每次執行先刪除舊資料表，確保乾淨的初始狀態
        cursor.execute("DROP TABLE IF EXISTS record;")
        cursor.execute("DROP TABLE IF EXISTS dept;")

        # 建立 dept 資料表（科別名稱、總名額、已預約人數）
        cursor.execute("""
            CREATE TABLE dept (
                subject TEXT PRIMARY KEY,
                quota INTEGER,
                booked_count INTEGER
            )
        """)

        # 建立 record 資料表（看診編號、患者名稱、預約科別、狀態）
        cursor.execute("""
            CREATE TABLE record (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT,
                dept TEXT,
                status TEXT
            )
        """)

        # 放入初始資料：內科、外科、牙科，預設額度 5 人，已預約 0 人
        initial_depts = [
            ('內科', 5, 0),
            ('外科', 5, 0),
            ('牙科', 5, 0)
        ]
        cursor.executemany("INSERT INTO dept (subject, quota, booked_count) VALUES (?, ?, ?)", initial_depts)

        conn.commit()
        print("✅ 資料庫建立成功，並已載入初始資料！")
    except Exception as e:
        print(f"❌ 建立資料庫發生錯誤: {e}")
    finally:
        conn.close()

# 執行初始化
init_db()

✅ 資料庫建立成功，並已載入初始資料！


In [3]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (3) 顯示資料表
import sqlite3
import pandas as pd

def show_tables():
    """使用 pandas 讀取並顯示目前的 dept 與 record 資料表內容"""
    conn = sqlite3.connect("clinic.db")
    try:
        print("=== 【dept 科別資料表】 ===")
        df_dept = pd.read_sql_query("SELECT * FROM dept", conn)
        display(df_dept)

        print("\n=== 【record 掛號記錄表】 ===")
        df_record = pd.read_sql_query("SELECT * FROM record", conn)
        display(df_record)
    except Exception as e:
        print(f"讀取資料表發生錯誤: {e}")
    finally:
        conn.close()

# 顯示目前資料表狀態
show_tables()

=== 【dept 科別資料表】 ===


,subject,quota,booked_count
0,內科,5,0
1,外科,5,0
2,牙科,5,0



=== 【record 掛號記錄表】 ===


,id,name,dept,status


In [4]:
# ============================================================
# 第 4 格：AI 解讀函式
# 提醒：回傳 (動作, 科別, 原始回覆)；不可直接相信 AI 的回覆
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (4) AI 解讀函式
def interpret_intent(user_input):
    """
    使用 Groq 模型分析使用者的話，提取出「動作」與「科別」。
    規定：temperature=0，失敗時回傳「未知」，並進行嚴格驗證。
    """
    prompt = f"""
    你是一個診所語意解析助理。請分析使用者的話，並嚴格只回傳兩個詞，中間用逗號隔開：`動作,科別`。
    動作只能是以下三種之一：掛號、取消掛號、查詢名額。
    科別只能是以下三種之一：內科、外科、牙科。
    如果使用者的話語無法對應上述任何一種動作與科別（例如只描述症狀、無關話題或未指明科別），請一律回傳：未知,未知。
    絕對不要提供任何醫療建議或推薦科別。

    使用者的話：{user_input}
    """

    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw_response = completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"AI 呼叫失敗: {e}")
        return "未知", "未知", "API錯誤"

    # 嚴格驗證機制：從回覆中對照允許清單，找不到就當作「未知」
    allowed_actions = ["掛號", "取消掛號", "查詢名額"]
    allowed_depts = ["內科", "外科", "牙科"]

    action = "未知"
    dept = "未知"

    parts = raw_response.split(",")
    if len(parts) >= 2:
        parsed_action = parts[0].strip()
        parsed_dept = parts[1].strip()

        if parsed_action in allowed_actions:
            action = parsed_action
        if parsed_dept in allowed_depts:
            dept = parsed_dept

    return action, dept, raw_response

In [5]:
# ============================================================
# 第 5 格：測試清單
# 提醒：至少 8 句，要與規格書第 7 節一致
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (5) 測試清單（包含正常、錯誤、邊界共 8 句）
test_cases = [
    ("我要掛內科", "掛號", "內科"),                              # 1. 正常
    ("查詢外科剩餘名額", "查詢名額", "外科"),                      # 2. 正常
    ("取消我的牙科預約", "取消掛號", "牙科"),                      # 3. 正常
    ("我要掛眼科", "掛號", "未知"),                                # 4. 錯誤（不在允許科別內）
    ("我最近一直咳嗽、胸悶，該看什麼科？", "未知", "未知"),        # 5. 錯誤（描述症狀/醫療建議）
    ("幫我取消內科預約", "取消掛號", "內科"),                      # 6. 正常（語意對應取消內科）
    ("內科名額已滿時我要掛內科", "掛號", "內科"),                  # 7. 邊界語意測試
    ("今天天氣真好，幫我推薦好吃的一家店", "未知", "未知")         # 8. 錯誤（無關話題）
]

def run_test_suite():
    print("=== 開始執行 AI 語意解讀測試清單 ===")
    pass_count = 0

    for i, (utterance, expected_action, expected_dept) in enumerate(test_cases, 1):
        action, dept, raw_resp = interpret_intent(utterance)

        is_match = (action == expected_action and dept == expected_dept)
        if is_match:
            pass_count += 1
            status_str = "【通過 ✅】"
        else:
            status_str = "【未符預期 ❌】"

        print(f"測試項目 {i}: {status_str}")
        print(f"  - 使用者說：{utterance}")
        print(f"  - AI 原始回覆：{raw_resp}")
        print(f"  - 系統解讀結果：動作 = {action} | 科別 = {dept}")
        print(f"  - 預期結果：動作 = {expected_action} | 科別 = {expected_dept}\n")

    print(f"🎯 測試完成！總共 {len(test_cases)} 句，通過 {pass_count} 句。")

# 執行測試清單
run_test_suite()

=== 開始執行 AI 語意解讀測試清單 ===
測試項目 1: 【通過 ✅】
  - 使用者說：我要掛內科
  - AI 原始回覆：掛號,內科
  - 系統解讀結果：動作 = 掛號 | 科別 = 內科
  - 預期結果：動作 = 掛號 | 科別 = 內科

測試項目 2: 【通過 ✅】
  - 使用者說：查詢外科剩餘名額
  - AI 原始回覆：查詢名額,外科
  - 系統解讀結果：動作 = 查詢名額 | 科別 = 外科
  - 預期結果：動作 = 查詢名額 | 科別 = 外科

測試項目 3: 【通過 ✅】
  - 使用者說：取消我的牙科預約
  - AI 原始回覆：取消掛號,牙科
  - 系統解讀結果：動作 = 取消掛號 | 科別 = 牙科
  - 預期結果：動作 = 取消掛號 | 科別 = 牙科

測試項目 4: 【未符預期 ❌】
  - 使用者說：我要掛眼科
  - AI 原始回覆：未知,未知
  - 系統解讀結果：動作 = 未知 | 科別 = 未知
  - 預期結果：動作 = 掛號 | 科別 = 未知

測試項目 5: 【通過 ✅】
  - 使用者說：我最近一直咳嗽、胸悶，該看什麼科？
  - AI 原始回覆：未知,未知
  - 系統解讀結果：動作 = 未知 | 科別 = 未知
  - 預期結果：動作 = 未知 | 科別 = 未知

測試項目 6: 【通過 ✅】
  - 使用者說：幫我取消內科預約
  - AI 原始回覆：取消掛號,內科
  - 系統解讀結果：動作 = 取消掛號 | 科別 = 內科
  - 預期結果：動作 = 取消掛號 | 科別 = 內科

測試項目 7: 【通過 ✅】
  - 使用者說：內科名額已滿時我要掛內科
  - AI 原始回覆：掛號,內科
  - 系統解讀結果：動作 = 掛號 | 科別 = 內科
  - 預期結果：動作 = 掛號 | 科別 = 內科

測試項目 8: 【通過 ✅】
  - 使用者說：今天天氣真好，幫我推薦好吃的一家店
  - AI 原始回覆：未知,未知
  - 系統解讀結果：動作 = 未知 | 科別 = 未知
  - 預期結果：動作 = 未知 | 科別 = 未知

🎯 測試完成！總共 8 句，通過 7 句。


In [6]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (6) 功能函式（掛號、取消掛號、查詢名額）
import sqlite3

def process_clinic_request(action, dept, patient_name="一般患者"):
    """
    根據解讀出的動作與科別，執行對應的資料庫查詢與更新邏輯。
    包含 fetchone() 的 None 檢查與 try/finally 確保連線關閉。
    """
    conn = sqlite3.connect("clinic.db")
    cursor = conn.cursor()

    try:
        # 1. 查詢名額
        if action == "查詢名額":
            cursor.execute("SELECT quota, booked_count FROM dept WHERE subject = ?", (dept,))
            row = cursor.fetchone()
            if row is None:
                return f"查詢失敗：找不到科別「{dept}」。"
            quota, booked_count = row
            remaining = quota - booked_count
            return f"【查詢結果】{dept} 目前總額度：{quota} 人，已約：{booked_count} 人，剩餘名額：{remaining} 人。"

        # 2. 掛號
        elif action == "掛號":
            cursor.execute("SELECT quota, booked_count FROM dept WHERE subject = ?", (dept,))
            row = cursor.fetchone()
            if row is None:
                return f"掛號失敗：找不到科別「{dept}」。"
            quota, booked_count = row

            if booked_count >= quota:
                return f"【掛號失敗】很抱歉，{dept} 今日名額已滿（已滿 {quota} 人）。"

            # 執行掛號：record 新增記錄，dept 的 booked_count + 1
            cursor.execute("INSERT INTO record (name, dept, status) VALUES (?, ?, ?)", (patient_name, dept, "有效"))
            cursor.execute("UPDATE dept SET booked_count = booked_count + 1 WHERE subject = ?", (dept,))
            conn.commit()

            record_id = cursor.lastrowid
            return f"【掛號成功】{dept} 掛號成功！您的看診編號為：#{record_id}"

        # 3. 取消掛號
        elif action == "取消掛號":
            cursor.execute("SELECT id FROM record WHERE dept = ? AND status = '有效' ORDER BY id DESC LIMIT 1", (dept,))
            row = cursor.fetchone()
            if row is None:
                return f"【取消失敗】找不到您在「{dept}」的有效掛號記錄，無法取消。"

            record_id = row[0]

            # 更新記錄狀態為「已取消」，對應科別 booked_count - 1
            cursor.execute("UPDATE record SET status = '已取消' WHERE id = ?", (record_id,))
            cursor.execute("UPDATE dept SET booked_count = booked_count - 1 WHERE subject = ?", (dept,))
            conn.commit()

            return f"【取消成功】已成功取消 {dept} 的預約（編號 #{record_id}），該科別名額已釋出。"

        else:
            return "【系統回應】無法辨識動作，系統不提供醫療建議或推薦科別。"

    except Exception as e:
        conn.rollback()
        return f"資料庫處理發生錯誤: {e}"
    finally:
        conn.close()

In [7]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# 提醒：最後呼叫 show_tables() 確認資料庫有變化
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (7) 主流程與完整流程測試
def clinic_system_pipeline(user_input, patient_name="王小明"):
    """整合 AI 意圖解讀與後端業務邏輯的主流程"""
    print(f"\n--- 模擬使用者輸入：「{user_input}」 ---")

    # 1. 透過 AI 解讀意圖
    action, dept, raw_resp = interpret_intent(user_input)
    print(f"🤖 AI 原始輸出: {raw_resp} -> 解析結果: 動作 = {action}, 對象 = {dept}")

    # 2. 檢查是否為未知
    if action == "未知" or dept == "未知":
        response = "【系統回應】無法辨識您的需求。本系統僅支援「內科、外科、牙科」的「掛號、取消掛號、查詢名額」，且絕對不提供任何醫療建議。"
        print(response)
        return response

    # 3. 執行後端業務邏輯
    response = process_clinic_request(action, dept, patient_name)
    print(response)
    return response

# 執行整合測試流程
print("=== 開始執行完整流程互動測試 ===")
clinic_system_pipeline("查詢內科還有多少名額")
clinic_system_pipeline("我要掛內科", "張三")
clinic_system_pipeline("查詢內科還有多少名額")
clinic_system_pipeline("取消我的內科預約")
clinic_system_pipeline("我肚子痛，應該看什麼科？")
clinic_system_pipeline("我要掛眼科")

# 再次顯示資料表確認資料庫變化
print("\n=== 測試後的資料表狀態 ===")
show_tables()

=== 開始執行完整流程互動測試 ===

--- 模擬使用者輸入：「查詢內科還有多少名額」 ---
🤖 AI 原始輸出: 查詢名額,內科 -> 解析結果: 動作 = 查詢名額, 對象 = 內科
【查詢結果】內科 目前總額度：5 人，已約：0 人，剩餘名額：5 人。

--- 模擬使用者輸入：「我要掛內科」 ---
🤖 AI 原始輸出: 掛號,內科 -> 解析結果: 動作 = 掛號, 對象 = 內科
【掛號成功】內科 掛號成功！您的看診編號為：#1

--- 模擬使用者輸入：「查詢內科還有多少名額」 ---
🤖 AI 原始輸出: 查詢名額,內科 -> 解析結果: 動作 = 查詢名額, 對象 = 內科
【查詢結果】內科 目前總額度：5 人，已約：1 人，剩餘名額：4 人。

--- 模擬使用者輸入：「取消我的內科預約」 ---
🤖 AI 原始輸出: 取消掛號,內科 -> 解析結果: 動作 = 取消掛號, 對象 = 內科
【取消成功】已成功取消 內科 的預約（編號 #1），該科別名額已釋出。

--- 模擬使用者輸入：「我肚子痛，應該看什麼科？」 ---
🤖 AI 原始輸出: 未知,未知 -> 解析結果: 動作 = 未知, 對象 = 未知
【系統回應】無法辨識您的需求。本系統僅支援「內科、外科、牙科」的「掛號、取消掛號、查詢名額」，且絕對不提供任何醫療建議。

--- 模擬使用者輸入：「我要掛眼科」 ---
🤖 AI 原始輸出: 未知,未知 -> 解析結果: 動作 = 未知, 對象 = 未知
【系統回應】無法辨識您的需求。本系統僅支援「內科、外科、牙科」的「掛號、取消掛號、查詢名額」，且絕對不提供任何醫療建議。

=== 測試後的資料表狀態 ===
=== 【dept 科別資料表】 ===


,subject,quota,booked_count
0,內科,5,0
1,外科,5,0
2,牙科,5,0



=== 【record 掛號記錄表】 ===


,id,name,dept,status
0,1,張三,內科,已取消


In [8]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：放在最後一格；GitHub 上看不到網頁，批改看的是第 5、7 格的結果
# ============================================================
# （把 AI 產生、你看過並理解的程式貼在這裡）
# (8) Gradio 網頁介面
import gradio as gr

def gradio_handler(user_input, patient_name):
    """Gradio 互動處理函式"""
    if not user_input.strip():
        return "請輸入您的需求或問題。"
    if not patient_name.strip():
        patient_name = "訪客"

    # 1. AI 意圖解讀
    action, dept, raw_resp = interpret_intent(user_input)

    # 2. 檢查是否為未知
    if action == "未知" or dept == "未知":
        return f"【AI 原始輸出】{raw_resp}\n【系統回應】無法辨識您的動作或科別。\n⚠️ 注意：本系統不提供醫療建議、不推薦科別，僅提供 內科、外科、牙科 的掛號、取消與查詢服務。"

    # 3. 執行資料庫動作
    result_msg = process_clinic_request(action, dept, patient_name)

    return f"【AI 原始輸出】{raw_resp}\n【系統解讀】動作：{action} | 科別：{dept}\n\n{result_msg}"

# 建立 Gradio Blocks 介面
with gr.Blocks(title="診所掛號系統") as demo:
    gr.Markdown("# 🏥 智慧診所掛號系統")
    gr.Markdown("歡迎使用診所掛號小幫手！您可以輸入像是「我要掛內科」、「查詢外科名額」或「取消牙科預約」。\n\n*⚠️ 宣告：本系統不提供醫療診斷或推薦科別。*")

    with gr.Row():
        with gr.Column():
            name_input = gr.Textbox(label="患者姓名", value="王小明", placeholder="請輸入您的姓名")
            query_input = gr.Textbox(label="您的需求", placeholder="例如：我要掛內科 / 查詢外科剩餘名額")
            submit_btn = gr.Button("送出查詢", variant="primary")
        with gr.Column():
            output_box = gr.Textbox(label="系統處理結果與 AI 回應", lines=7)

    submit_btn.click(
        fn=gradio_handler,
        inputs=[query_input, name_input],
        outputs=output_box
    )

# 啟動 Gradio 網頁（在 Colab 內直接呈現）
demo.launch(inline=True, share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

# ✅ 交件前檢查清單（每一項都要打勾）

- [ ] 程式裡**沒有**任何 `gsk_` 開頭的金鑰，也沒有 print 出金鑰
- [ ] 規格書第 1～7 節都填完，測試清單至少 8 句
- [ ] 選單「執行階段 → 重新啟動並執行全部」，**全部跑完沒有紅色錯誤**
- [ ] 每一格左邊的執行編號是**連續遞增**的（1、2、3…）
- [ ] 選單「編輯 → 筆記本設定」，**沒有**勾選「儲存筆記本時忽略程式碼儲存格輸出內容」
- [ ] 選單「檔案 → 在 GitHub 中儲存副本」，存到**公開（Public）**的 repo
- [ ] 用無痕視窗打開 GitHub 上的檔案，確認看得到程式和執行結果